# 01 — Pilot: FP16 baseline on 50 GSM8K + 50 TriviaQA

**Goal:** check that prompts, answer extraction, scoring and confidence all work
*before* any quantization or full runs.

Settings: Accelerator = **GPU T4 x2** (only one GPU is used), Internet = **On**.

What to check at the end:
1. GSM8K FP16 accuracy is roughly 65–75% (Li et al. report 68.23 for Qwen2.5-1.5B-Instruct).
2. Almost every GSM8K output has a `\boxed{}` answer.
3. TriviaQA: compare `em` vs `contains` and read the printed examples to choose the scoring rule.
4. The time per question, to estimate the full run.

In [ ]:
# 1. Record library versions (write these in your report; keep them fixed for all runs)
import sys, torch, transformers, datasets
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU!")

In [ ]:
# 2. Settings
MODEL_ID   = "Qwen/Qwen2.5-1.5B-Instruct"
N_PILOT    = 50          # questions per dataset for the pilot
SEED       = 42          # fixed seed for the TriviaQA subset
BATCH_GSM  = 16
BATCH_TQA  = 32
MAX_NEW_GSM = 1024
MAX_NEW_TQA = 32
OUT_DIR    = "/kaggle/working"
TAG        = "fp16"      # later: "rtn4", "gptq4", ...

In [ ]:
# 3. Helper functions: answer extraction and scoring (tested offline)
import re
import string
import unicodedata
from fractions import Fraction


# ---------- GSM8K ----------
def gsm8k_gold(answer_text):
    """Gold answer is the number after '####'."""
    return answer_text.split("####")[-1].strip().replace(",", "")


def last_boxed(text):
    """Content of the last \\boxed{...}, handling nested braces."""
    start = text.rfind("\\boxed{")
    if start == -1:
        return None
    i = start + len("\\boxed{")
    depth = 1
    out = []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
        out.append(c)
        i += 1
    return "".join(out) if depth == 0 else None


def to_number(s):
    """Turn '1,234', '$12.50', '50%', '\\frac{11}{2}', '11/2' into a float. None if impossible."""
    if s is None:
        return None
    s = s.strip()
    s = s.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    m = re.search(r"\\frac\{\s*(-?[\d.]+)\s*\}\{\s*(-?[\d.]+)\s*\}", s)
    if m:
        try:
            return float(Fraction(m.group(1)) / Fraction(m.group(2)))
        except (ValueError, ZeroDivisionError):
            return None
    s = re.sub(r"\\text\{([^}]*)\}", r"\1", s)
    s = s.replace(",", "").replace("$", "").replace("%", "").replace("\\", "")
    m = re.search(r"-?\d+(?:\.\d+)?\s*/\s*-?\d+(?:\.\d+)?", s)
    if m:
        try:
            a, b = m.group(0).split("/")
            return float(Fraction(a.strip()) / Fraction(b.strip()))
        except (ValueError, ZeroDivisionError):
            return None
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group(0)) if m else None


def gsm8k_pred(output):
    """Prediction = last \\boxed{} if present, otherwise the last number in the text."""
    boxed = last_boxed(output)
    if boxed is not None:
        return boxed, "boxed"
    nums = re.findall(r"-?\d[\d,]*(?:\.\d+)?", output)
    return (nums[-1] if nums else None), "last_number"


def gsm8k_correct(pred, gold):
    p, g = to_number(pred), to_number(gold)
    return p is not None and g is not None and abs(p - g) < 1e-4


# ---------- TriviaQA ----------
def normalize_answer(s):
    """Standard SQuAD/TriviaQA normalization: lowercase, no accents, no punctuation, no articles."""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))   # Río -> Rio
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def trivia_pred(output):
    """Keep the first line and drop a leading 'Answer:'."""
    line = output.strip().split("\n")[0].strip()
    line = re.sub(r"^(final\s+)?answer\s*:\s*", "", line, flags=re.I)
    return line


def trivia_scores(pred, aliases):
    """em: normalized prediction equals an alias.
    contains: an alias appears as whole words inside a short prediction (<= 10 words)."""
    p = normalize_answer(pred)
    norm_aliases = {normalize_answer(a) for a in aliases if a.strip()}
    em = p in norm_aliases
    contains = len(p.split()) <= 10 and any(
        a and re.search(r"(^| )" + re.escape(a) + r"( |$)", p) for a in norm_aliases
    )
    return em, contains

In [ ]:
# 4. Load data
from datasets import load_dataset

gsm = load_dataset("openai/gsm8k", "main", split="test")
tqa = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")
print(len(gsm), "GSM8K test questions |", len(tqa), "TriviaQA validation questions")

gsm_pilot = gsm.select(range(N_PILOT))
tqa_pilot = tqa.shuffle(seed=SEED).select(range(N_PILOT))

In [ ]:
# 5. Load the model in FP16 on one GPU
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"                       # needed for batched generation
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map={"": 0})   # T4 has no bf16
model.eval()
EOS_IDS = set(model.generation_config.eos_token_id
              if isinstance(model.generation_config.eos_token_id, list)
              else [model.generation_config.eos_token_id])
print("EOS ids:", EOS_IDS, "| pad id:", tok.pad_token_id)

In [ ]:
# 6. Prompts (fixed for every model)
def gsm_prompt(q):
    msgs = [{"role": "user",
             "content": q + "\nPlease reason step by step, and put your final answer within \\boxed{}."}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def tqa_prompt(q):
    msgs = [{"role": "user",
             "content": "Answer the following question with only the answer, in a few words.\n"
                        "Question: " + q}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print(gsm_prompt("What is 2+3?"))

In [ ]:
# 7. Greedy generation + confidence (mean and min token log-probability)
#    - Confidence is recorded step by step (one number per token), so it uses
#      almost no GPU memory.
#    - Pure greedy decoding: Qwen's default repetition penalty is turned OFF,
#      because loops/repetition are errors we want to see, not hide.
import time
from transformers import LogitsProcessor, LogitsProcessorList

class RecordLogprob(LogitsProcessor):
    # Greedy decoding picks the most likely token, so its log-probability
    # is the maximum of the log-softmax at each step.
    def __init__(self):
        self.steps = []
    def __call__(self, input_ids, scores):
        self.steps.append(torch.log_softmax(scores.float(), dim=-1).max(dim=-1).values)
        return scores

@torch.no_grad()
def generate(prompts, max_new, batch_size):
    results = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        enc = tok(batch, return_tensors="pt", padding=True).to(model.device)
        rec = RecordLogprob()
        seqs = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                              temperature=None, top_p=None, top_k=None,
                              repetition_penalty=1.0,
                              logits_processor=LogitsProcessorList([rec]),
                              pad_token_id=tok.pad_token_id)
        gen = seqs[:, enc["input_ids"].shape[1]:]
        logp = torch.stack(rec.steps, dim=1).cpu()
        for j in range(gen.shape[0]):
            ids = gen[j].tolist()
            # keep tokens up to and including the first EOS
            n = next((k + 1 for k, t in enumerate(ids) if t in EOS_IDS), len(ids))
            lp = logp[j, :n].float().cpu()
            results.append({
                "output": tok.decode(ids[:n], skip_special_tokens=True),
                "n_tokens": n,
                "hit_limit": n == max_new and ids[n - 1] not in EOS_IDS,
                "mean_logprob": lp.mean().item(),
                "min_logprob": lp.min().item(),
            })
        print(f"  {min(i + batch_size, len(prompts))}/{len(prompts)} done")
    return results

In [ ]:
# 8. Run GSM8K pilot
t0 = time.time()
gsm_out = generate([gsm_prompt(x["question"]) for x in gsm_pilot], MAX_NEW_GSM, BATCH_GSM)
gsm_time = time.time() - t0

gsm_rows = []
for k, (ex, r) in enumerate(zip(gsm_pilot, gsm_out)):
    gold = gsm8k_gold(ex["answer"])
    pred, how = gsm8k_pred(r["output"])
    gsm_rows.append({"id": k, "question": ex["question"], "gold": gold,
                     "pred": pred, "extracted_by": how,
                     "correct": gsm8k_correct(pred, gold), **r})
acc = sum(r["correct"] for r in gsm_rows) / len(gsm_rows)
boxed = sum(r["extracted_by"] == "boxed" for r in gsm_rows)
cut = sum(r["hit_limit"] for r in gsm_rows)
print(f"GSM8K FP16 accuracy: {acc:.1%} | boxed answers: {boxed}/{len(gsm_rows)} "
      f"| hit token limit: {cut} | {gsm_time/len(gsm_rows):.2f} s/question")

In [ ]:
# 9. Run TriviaQA pilot
t0 = time.time()
tqa_out = generate([tqa_prompt(x["question"]) for x in tqa_pilot], MAX_NEW_TQA, BATCH_TQA)
tqa_time = time.time() - t0

tqa_rows = []
for ex, r in zip(tqa_pilot, tqa_out):
    aliases = ex["answer"]["aliases"] + [ex["answer"]["value"]]
    pred = trivia_pred(r["output"])
    em, contains = trivia_scores(pred, aliases)
    tqa_rows.append({"id": ex["question_id"], "question": ex["question"],
                     "gold": ex["answer"]["value"], "pred": pred,
                     "em": em, "contains": contains, **r})
em_acc = sum(r["em"] for r in tqa_rows) / len(tqa_rows)
ct_acc = sum(r["contains"] for r in tqa_rows) / len(tqa_rows)
print(f"TriviaQA FP16  EM: {em_acc:.1%} | contains: {ct_acc:.1%} "
      f"| {tqa_time/len(tqa_rows):.2f} s/question")

In [ ]:
# 10. Look at examples (this is the real point of the pilot)
print("=== GSM8K: wrong answers ===")
for r in [r for r in gsm_rows if not r["correct"]][:5]:
    print(f"gold={r['gold']} pred={r['pred']} ({r['extracted_by']}) tokens={r['n_tokens']}")
    print(r["output"][-300:].replace("\n", " "), "\n")

print("=== TriviaQA: EM and contains disagree (check which is right) ===")
for r in [r for r in tqa_rows if r["em"] != r["contains"]][:10]:
    print(f"Q: {r['question']}\n  gold={r['gold']!r}  pred={r['pred']!r}  em={r['em']} contains={r['contains']}")

print("\n=== TriviaQA: both wrong ===")
for r in [r for r in tqa_rows if not r["contains"]][:10]:
    print(f"Q: {r['question']}\n  gold={r['gold']!r}  pred={r['pred']!r}")

In [ ]:
# 11. Save outputs (.jsonl, one line per question) + time estimate for the full run
import json
for name, rows in [("gsm8k", gsm_rows), ("triviaqa", tqa_rows)]:
    path = f"{OUT_DIR}/pilot_{name}_{TAG}.jsonl"
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print("saved", path)

full_gsm_h = gsm_time / N_PILOT * 1319 / 3600
full_tqa_h = tqa_time / N_PILOT * 2000 / 3600
print(f"Estimated full run per model: GSM8K {full_gsm_h:.1f} h, TriviaQA (2,000) {full_tqa_h:.1f} h")